In [0]:
from pyspark.sql.functions import col, lpad, trim, date_format, from_utc_timestamp
from delta.tables import DeltaTable

In [0]:
dbutils.widgets.text("year", "None", "Year (YYYY)")
dbutils.widgets.text("month", "None", "Month (MM)")

year = dbutils.widgets.get("year")
month = dbutils.widgets.get("month")

In [0]:
source_table = "oftalmo_sus.01_bronze.tb_datasus_sih"
target_table = "oftalmo_sus.02_silver.tb_fato_sih"

print("🔄 Iniciando higienização da camada Silver para o SIH (Hospitalar/Cirurgias)...")

In [0]:
# 2. Leitura da Bronze
df_bronze_sih = spark.read.table(source_table).filter(f"batch_processing = '{year}{month}'")
#display(df_bronze_sih.limit(10))

In [0]:
# 3. Transformação (Data Quality): Recuperando os 10 dígitos do Procedimento
df_silver_sih = df_bronze_sih.withColumn(
    "SP_PROC",
    lpad(trim(col("SP_PROCREA").cast("bigint").cast("string")), 10, "0")
)

In [0]:
# 4. Gravação na Silver
(
    df_silver_sih.write 
    .format("delta")
    .mode("append")
    .partitionBy("batch_processing")
    .option("mergeSchema", "true")
    .saveAsTable(target_table)
)

print(f"✅ SIH padronizado com sucesso na Silver em: {target_table}")

In [0]:
delta_table = DeltaTable.forName(spark, target_table)

history_df = delta_table.history()

display(
    history_df
    .orderBy(col("version").desc())
    .limit(1)
    .select(
        col("version"),
        date_format(from_utc_timestamp(col("timestamp"), "America/Sao_Paulo"), "dd-MM-yyyy HH:mm:ss").alias("timestamp"),
        col("operation"),
        col("operationMetrics.insert").alias("insert"),
        col("operationMetrics.delete").alias("delete"),
        col("operationMetrics.update").alias("update"),
        col("operationMetrics.numOutputRows").alias("num_rows")
    )
)